# Exploratory Data Analysis (EDA): Pola Peminjaman & Prediksi Keterlambatan Buku Perpustakaan
**Tugas Ujian Tengah Semester (UTS) — Mata Kuliah Data Science (PSIKS2W002)**
* **Nama Mahasiswa:** Arif Tri Wicaksono
* **NIM:** 26MA4D015
* **Program Studi:** Magister Ilmu Komputer, Universitas Amikom Purwokerto
* **Dosen Pengampu:** Giat Karyono, M.Kom.

---

## Bab 1: Pendahuluan & Memahami Masalah (*Understanding the Problem*)
* **Latar Belakang & Tujuan Utama:** Mengeksplorasi pola sirkulasi peminjaman buku perpustakaan (`library_data.csv`) dan mengidentifikasi faktor utama pemicu keterlambatan pengembalian (`overdue_status`).
* **Tipe Masalah:** *Supervised Learning* — Klasifikasi Biner (`0` = Tepat Waktu, `1` = Terlambat).
* **Metrik Keberhasilan Utama:** *Recall* (meminimalkan kasus keterlambatan yang terlewat) dan *F1-Score*.

## Bab 2 & 3: Memuat & Memeriksa Data (*Understanding, Importing & Inspecting Data*)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Pengaturan gaya visualisasi
sns.set_theme(style="whitegrid")

# 3.1 Memuat dataset
df = pd.read_csv('library_data.csv')
print("Dimensi Dataset (Baris, Kolom):", df.shape)
df.head()

In [ ]:
# 3.2 Inspeksi tipe data dan informasi umum
print("--- INFORMASI TIPE DATA ---")
df.info()

print("\n--- RINGKASAN STATISTIK AWAL ---")
display(df.describe(include='all'))

## Bab 4: Penanganan Data yang Hilang (*Handling Missing Data*) & Kualitas Data

In [ ]:
# 4.1 Identifikasi Missing Values dan Data Duplikat
missing_df = pd.DataFrame({
    'Jumlah Missing': df.isnull().sum(),
    'Persentase (%)': (df.isnull().sum() / len(df)) * 100
})
print("--- CEK MISSING VALUES ---")
display(missing_df)
print("Jumlah baris duplikat:", df.duplicated().sum())

## Bab 5 & 6: Eksplorasi Karakteristik Data & Transformasi (*Feature Engineering*)

In [ ]:
# 6.4 Pembuatan Fitur Baru (Feature Engineering)
df['borrow_date'] = pd.to_datetime(df['borrow_date'])
df['return_date'] = pd.to_datetime(df['return_date'])

# Ekstraksi durasi peminjaman dalam hari
df['loan_duration'] = (df['return_date'] - df['borrow_date']).dt.days
df['borrow_month'] = df['borrow_date'].dt.month
df['is_weekend_borrow'] = df['borrow_date'].dt.dayofweek.isin([5, 6]).astype(int)

# Binning durasi pinjam
df['duration_category'] = pd.cut(df['loan_duration'], bins=[0, 7, 14, 21], labels=['Pendek', 'Sedang', 'Lama'])

# 5.1 & 5.2 Statistik Deskriptif, Skewness, dan Kurtosis
print("--- STATISTIK DURASI PEMINJAMAN (HARI) ---")
print(df['loan_duration'].describe())
print(f"Skewness: {skew(df['loan_duration']):.4f}")
print(f"Kurtosis: {kurtosis(df['loan_duration']):.4f}")

In [ ]:
# 6.2 & 6.3 Scaling & Categorical Encoding
scaler_minmax = MinMaxScaler()
df['loan_duration_minmax'] = scaler_minmax.fit_transform(df[['loan_duration']])

scaler_std = StandardScaler()
df['loan_duration_std'] = scaler_std.fit_transform(df[['loan_duration']])

# One-Hot Encoding untuk variabel kategorikal nominal
df_encoded = pd.get_dummies(df, columns=['user_role', 'book_category'], drop_first=False)
df_encoded['reservation_status'] = df_encoded['reservation_status'].astype(int)

print("Dimensi setelah Encoding & Feature Engineering:", df_encoded.shape)
df_encoded.head(3)

## Bab 7: Visualisasi Hubungan Data (*Univariate, Bivariate & Multivariate*)

In [ ]:
# 7.1 Visualisasi Univariat
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

ax1 = sns.countplot(data=df, x='overdue_status', ax=axes[0], palette=['#4E79A7', '#E15759'])
axes[0].set_title('Distribusi Status Keterlambatan', fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Tepat Waktu (0)', 'Terlambat (1)'])
for p in ax1.patches:
    ax1.annotate(f'{int(p.get_height())} ({p.get_height()/len(df)*100:.1f}%)',
                 (p.get_x() + p.get_width()/2., p.get_height()), ha='center', va='bottom', fontweight='bold')

ax2 = sns.countplot(data=df, x='user_role', order=['student', 'staff', 'faculty'], ax=axes[1], palette='Blues_d')
axes[1].set_title('Distribusi Peran Peminjam', fontweight='bold')
for p in ax2.patches:
    ax2.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()), ha='center', va='bottom', fontweight='bold')

ax3 = sns.countplot(data=df, x='book_category', order=df['book_category'].value_counts().index, ax=axes[2], palette='viridis')
axes[2].set_title('Distribusi Kategori Buku', fontweight='bold')
axes[2].tick_params(axis='x', rotation=15)
for p in ax3.patches:
    ax3.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 7.2 & 7.3 Bivariate Analysis & Correlation Heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='user_role', y='loan_duration', hue='overdue_status', ax=axes[0], palette=['#4E79A7', '#E15759'])
axes[0].set_title('Durasi Pinjam vs Keterlambatan per Peran', fontweight='bold')
handles, _ = axes[0].get_legend_handles_labels()
axes[0].legend(handles, ['Tepat Waktu (0)', 'Terlambat (1)'], title='Status')

corr_cols = ['loan_duration', 'overdue_status', 'borrow_month', 'reservation_status', 'user_role_faculty', 'user_role_student']
sns.heatmap(df_encoded[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('Heatmap Korelasi Antar Variabel', fontweight='bold')

plt.tight_layout()
plt.show()

## Bab 8: Penanganan Pencilan (*Handling Outliers*)

In [ ]:
# 8.1 Deteksi Outliers dengan Metode IQR
Q1 = df['loan_duration'].quantile(0.25)
Q3 = df['loan_duration'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['loan_duration'] < lower_bound) | (df['loan_duration'] > upper_bound)]
print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Batas Bawah: {lower_bound} hari | Batas Atas: {upper_bound} hari")
print(f"Jumlah Pencilan Terdeteksi: {len(outliers)} baris")

plt.figure(figsize=(8, 3))
sns.boxplot(x=df['loan_duration'], color='#76B7B2')
plt.title('Boxplot Deteksi Pencilan (Outliers) Durasi Peminjaman', fontweight='bold')
plt.xlabel('Durasi Peminjaman (Hari)')
plt.show()

## Bab 9: Kesimpulan & Rekomendasi Model (*Communicate Findings & Insights*)
1. **Tingkat Keterlambatan Tinggi (62,7%):** Mayoritas peminjaman melewati batas waktu standar 7 hari (korelasi $r = 0,83$ antara `loan_duration` dan `overdue_status`).
2. **Segmen Risiko:** Kelompok *faculty* (64,63%) dan kategori buku *Science* (66,18%) memiliki tingkat keterlambatan tertinggi.
3. **Rekomendasi Model:** Menggunakan **Random Forest Classifier** atau **XGBoost Classifier** pada tahap pemodelan prediktif berikutnya.